In [2]:
import os
import shutil
from pathlib import Path

# Combine parallel inferred cNMF component results into one run

In [2]:

def rename_and_move_files_NMF(file_name_input, file_name_output, source_folder, destination_folder, len = 10):

    # Create destination folder if it doesn't exist
    Path(destination_folder).mkdir(parents=True, exist_ok=True)
    
    processed_files = []
    
    # Process NMF files for i 
    for i in range(len):
        source_filename = f"{file_name_input}_{i}.df.npz"
        source_path = os.path.join(source_folder, source_filename)
        
        # Check if source file exists
        if os.path.exists(source_path):
            
            # Create new filename
            new_filename = f"{file_name_output}_{i}.df.npz"
            destination_path = os.path.join(destination_folder, new_filename)
            
            try:
                # Copy and rename file to destination
                shutil.copy2(source_path, destination_path)
                print(f"Successfully processed: {source_filename} -> {new_filename}")
                processed_files.append((source_filename, new_filename))
                
            except Exception as e:
                print(f"Error processing {source_filename}: {e}")
        else:
            print(f"File not found: {source_filename}")

        # process cNMF files for i
        for i in range(len):
            source_filename = f"{file_name_output}_{i}.df.npz"
    
    return processed_files

def rename_all_NMF(file_name_input, file_name_output, source_folder, destination_folder, len = 10, components = [30, 50, 60, 80, 100, 200, 250, 300]):

    for k in components:

        file_name_input_new = f"{file_name_input}_{k}.spectra.k_{k}.iter"
        file_name_output_new = f"{file_name_output}.spectra.k_{k}.iter"

        source_folder_new = f"{source_folder}_{k}/cnmf_tmp"

        rename_and_move_files_NMF(file_name_input_new, file_name_output_new, source_folder_new, destination_folder)


In [4]:
source_dir_base = "/oak/stanford/groups/engreitz/Users/ymo/Ronghao_100K_sample/Results/120825_100k_cells_10iter_allhvg_torch_halsvar_batch_e7_h100_par/120825_100k_cells_10iter_allhvg_torch_halsvar_batch_e7_h100_par"
dest_dir = "/oak/stanford/groups/engreitz/Users/ymo/Ronghao_100K_sample/Results/120825_100k_cells_10iter_allhvg_torch_halsvar_batch_e7_h100_par/120825_100k_cells_10iter_allhvg_torch_halsvar_batch_e7_h100_par_all/cnmf_tmp"

file_name_input = "120825_100k_cells_10iter_allhvg_torch_halsvar_batch_e7_h100_par"
file_name_output =  "120825_100k_cells_10iter_allhvg_torch_halsvar_batch_e7_h100_par_all"

rename_all_NMF(file_name_input,file_name_output,source_dir_base, dest_dir, components = [30, 50, 60, 80, 100, 200, 250, 300] )

Successfully processed: 120825_100k_cells_10iter_allhvg_torch_halsvar_batch_e7_h100_par_30.spectra.k_30.iter_0.df.npz -> 120825_100k_cells_10iter_allhvg_torch_halsvar_batch_e7_h100_par_all.spectra.k_30.iter_0.df.npz
Successfully processed: 120825_100k_cells_10iter_allhvg_torch_halsvar_batch_e7_h100_par_30.spectra.k_30.iter_1.df.npz -> 120825_100k_cells_10iter_allhvg_torch_halsvar_batch_e7_h100_par_all.spectra.k_30.iter_1.df.npz
Successfully processed: 120825_100k_cells_10iter_allhvg_torch_halsvar_batch_e7_h100_par_30.spectra.k_30.iter_2.df.npz -> 120825_100k_cells_10iter_allhvg_torch_halsvar_batch_e7_h100_par_all.spectra.k_30.iter_2.df.npz
Successfully processed: 120825_100k_cells_10iter_allhvg_torch_halsvar_batch_e7_h100_par_30.spectra.k_30.iter_3.df.npz -> 120825_100k_cells_10iter_allhvg_torch_halsvar_batch_e7_h100_par_all.spectra.k_30.iter_3.df.npz
Successfully processed: 120825_100k_cells_10iter_allhvg_torch_halsvar_batch_e7_h100_par_30.spectra.k_30.iter_4.df.npz -> 120825_100k_cel

# Comile consensus files

In [3]:
# Transfer over consensus step
def consolidate_parallel_files(base_name, source_base_dir, dest_dir, 
                             k_values=[30, 50, 60, 80, 100, 200, 250, 300]):
    """
    Consolidates files from parallel cNMF runs into a single directory.
    
    Parameters:
    - base_name: Base filename (without _K suffix)
    - source_base_dir: Base directory containing the _K subdirectories  
    - dest_dir: Destination directory for consolidated files
    - k_values: List of K values to process
    
    Files to copy (excluding k_selection files):
    - clustering files (.png)
    - gene_spectra_score files (.txt)
    - gene_spectra_tpm files (.txt) 
    - overdispersed_genes files (.txt)
    - spectra consensus files (.txt)
    - starcat_spectra files (.txt)
    - usages consensus files (.txt)
    """
    
    # Create destination folder if it doesn't exist
    Path(dest_dir).mkdir(parents=True, exist_ok=True)
    
    processed_files = []
    skipped_files = []
    
    # File patterns to include (exclude k_selection files)
    include_patterns = [
        'clustering',
        'gene_spectra_score', 
        'gene_spectra_tpm',
        'spectra',  # includes spectra consensus files
        'starcat_spectra',
        'usages.k_'    # includes usages consensus files
    ]
    
    # File patterns to exclude
    exclude_patterns = [
        'overdispersed_genes'
        'k_selection',
        'config_',
        '.yml'
    ]
    
    for k in k_values:
        source_dir = f"{source_base_dir}_{k}"
        
        if not os.path.exists(source_dir):
            print(f"Warning: Directory {source_dir} not found, skipping K={k}")
            continue
            
        print(f"Processing K={k} from {source_dir}")
        
        # Get all files in the source directory
        try:
            files = os.listdir(source_dir)
        except OSError as e:
            print(f"Error reading directory {source_dir}: {e}")
            continue
            
        for filename in files:
            # Skip if file matches exclude patterns
            if any(pattern in filename for pattern in exclude_patterns):
                skipped_files.append(f"{source_dir}/{filename}")
                continue
                
            # Only process files that match include patterns
            if any(pattern in filename for pattern in include_patterns):
                source_path = os.path.join(source_dir, filename)
                
                # Create new filename by removing the _K suffix from the base name
                # Example: "base_30.clustering.k_30.dt_0_4.png" -> "base.clustering.k_30.dt_0_4.png"
                if filename.startswith(f"{base_name}_{k}."):
                    new_filename = filename.replace(f"{base_name}_{k}.", f"{base_name}.")
                else:
                    new_filename = filename
                    
                dest_path = os.path.join(dest_dir, new_filename)
                
                try:
                    shutil.copy2(source_path, dest_path)
                    print(f"  Copied: {filename} -> {new_filename}")
                    processed_files.append((source_path, dest_path))
                    
                except Exception as e:
                    print(f"  Error copying {filename}: {e}")
            else:
                skipped_files.append(f"{source_dir}/{filename}")
    
    print(f"\nSummary:")
    print(f"  Processed files: {len(processed_files)}")
    print(f"  Skipped files: {len(skipped_files)}")
    print(f"  Files saved to: {dest_dir}")
    
    if skipped_files:
        print(f"\nSkipped files (k_selection, config, etc.):")
        for f in skipped_files[:10]:  # Show first 10 skipped files
            print(f"  {f}")
        if len(skipped_files) > 10:
            print(f"  ... and {len(skipped_files) - 10} more")
    
    return processed_files, skipped_files

In [4]:
# Example usage with your data
base_name = "120825_100k_cells_10iter_allhvg_torch_halsvar_batch_e7_h100_par"
source_base_dir = "/oak/stanford/groups/engreitz/Users/ymo/Ronghao_100K_sample/Results/120825_100k_cells_10iter_allhvg_torch_halsvar_batch_e7_h100_par/120825_100k_cells_10iter_allhvg_torch_halsvar_batch_e7_h100_par"
dest_dir = "/oak/stanford/groups/engreitz/Users/ymo/Ronghao_100K_sample/Results/120825_100k_cells_10iter_allhvg_torch_halsvar_batch_e7_h100_par/consolidated_results"

# Run the consolidation
processed, skipped = consolidate_parallel_files(base_name, source_base_dir, dest_dir)

Processing K=30 from /oak/stanford/groups/engreitz/Users/ymo/Ronghao_100K_sample/Results/120825_100k_cells_10iter_allhvg_torch_halsvar_batch_e7_h100_par/120825_100k_cells_10iter_allhvg_torch_halsvar_batch_e7_h100_par_30
  Copied: 120825_100k_cells_10iter_allhvg_torch_halsvar_batch_e7_h100_par_30.gene_spectra_tpm.k_30.dt_0_8.txt -> 120825_100k_cells_10iter_allhvg_torch_halsvar_batch_e7_h100_par.gene_spectra_tpm.k_30.dt_0_8.txt
  Copied: 120825_100k_cells_10iter_allhvg_torch_halsvar_batch_e7_h100_par_30.usages.k_30.dt_0_8.consensus.txt -> 120825_100k_cells_10iter_allhvg_torch_halsvar_batch_e7_h100_par.usages.k_30.dt_0_8.consensus.txt
  Copied: 120825_100k_cells_10iter_allhvg_torch_halsvar_batch_e7_h100_par_30.starcat_spectra.k_30.dt_0_4.txt -> 120825_100k_cells_10iter_allhvg_torch_halsvar_batch_e7_h100_par.starcat_spectra.k_30.dt_0_4.txt
  Copied: 120825_100k_cells_10iter_allhvg_torch_halsvar_batch_e7_h100_par_30.gene_spectra_tpm.k_30.dt_2_0.txt -> 120825_100k_cells_10iter_allhvg_torch_h

# combine 2 different cNMF into one

In [3]:
def rename_and_move_files(k, file_name_input, file_name_output, source_dir, dest_dir, len = 10, second = False):
    
    # Create destination folder if it doesn't exist
    Path(dest_dir).mkdir(parents=True, exist_ok=True)
    
    processed_files = []
    
    # Process files for i 
    for i in range(len):

        source_file = f"{file_name_input}.spectra.k_{k}.iter_{i}.df.npz"
        source_path = os.path.join(source_dir, source_file )

        # Check if source file exists
        if os.path.exists(source_dir): 

            if second: 
                i = i + 10
            
            # Create new filename
            new_filename = f"{file_name_output}.spectra.k_{k}.iter_{i}.df.npz"
            destination_path = os.path.join(dest_dir, new_filename)
            
            try:
                # Copy and rename file to destination
                shutil.copy2(source_path, destination_path)
                print(f"Successfully processed: {source_file} -> {new_filename}")
                processed_files.append((source_file , new_filename))
                
            except Exception as e:
                print(f"Error processing {file_name_input}: {e}")
        else:
            print(f"File not found: {file_name_input}")
            print(source_path)
    
    return processed_files


def rename_all(file_name_input, file_name_output, source_dir, dest_dir, components = [30, 50, 60, 80, 100, 200, 250, 300], second = False):

    for k in components:

        data = rename_and_move_files(k, file_name_input, file_name_output, source_dir, dest_dir, second = second)


In [6]:
source_dir = "/oak/stanford/groups/engreitz/Users/ymo/NMF_re-inplementing/Results/torch-cNMF_evaluation/091625_100k_cells_10iter_torch_halsvar_online_e7/cnmf_tmp"
dest_dir = "/oak/stanford/groups/engreitz/Users/ymo/NMF_re-inplementing/Results/Consensus_across_methods/online_halsvar_cd_e7/cnmf_tmp"

file_name_input = "091625_100k_cells_10iter_torch_halsvar_online_e7"
file_name_output =  "online_halsvar_cd_e7"

rename_all(file_name_input,file_name_output, source_dir, dest_dir, components = [30, 50, 60, 80, 100, 200, 250, 300], second = True)


Successfully processed: 091625_100k_cells_10iter_torch_halsvar_online_e7.spectra.k_30.iter_0.df.npz -> online_halsvar_cd_e7.spectra.k_30.iter_10.df.npz
Successfully processed: 091625_100k_cells_10iter_torch_halsvar_online_e7.spectra.k_30.iter_1.df.npz -> online_halsvar_cd_e7.spectra.k_30.iter_11.df.npz
Successfully processed: 091625_100k_cells_10iter_torch_halsvar_online_e7.spectra.k_30.iter_2.df.npz -> online_halsvar_cd_e7.spectra.k_30.iter_12.df.npz
Successfully processed: 091625_100k_cells_10iter_torch_halsvar_online_e7.spectra.k_30.iter_3.df.npz -> online_halsvar_cd_e7.spectra.k_30.iter_13.df.npz
Successfully processed: 091625_100k_cells_10iter_torch_halsvar_online_e7.spectra.k_30.iter_4.df.npz -> online_halsvar_cd_e7.spectra.k_30.iter_14.df.npz
Successfully processed: 091625_100k_cells_10iter_torch_halsvar_online_e7.spectra.k_30.iter_5.df.npz -> online_halsvar_cd_e7.spectra.k_30.iter_15.df.npz
Successfully processed: 091625_100k_cells_10iter_torch_halsvar_online_e7.spectra.k_30.it